# A8: RAG System — UIUC-Med Medical Triage Knowledge Base

**Team:** Group 8 — UIUC Med  
**Generation Model:** `llama-3.3-70b-versatile` via Groq (best model from A7)  
**Embedding Models:** `all-MiniLM-L6-v2` (small, 384-dim), `all-mpnet-base-v2` (medium, 768-dim), `all-roberta-large-v1` (large, 1024-dim)

## Pipeline Overview
```
Data → Chunking → Embeddings → Vector Store → Retrieval → LLM → Output
```

## Setup — Install Dependencies

In [1]:
%pip install -q sentence-transformers groq python-dotenv numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import time
import json
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from groq import Groq

# load the api keys from .env (place GROQ_API_KEY in UIUC-Med/.env)
load_dotenv(dotenv_path="../../../.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")



Groq client ready.
Generation model: llama-3.3-70b-versatile


---
## Part 1 — Build the RAG Pipeline

### Step 1.1 — Define the Knowledge Base

**Dataset:** Medical Triage & Health FAQ Knowledge Base  
This dataset was chosen because it directly supports the UIUC-Med app, which classifies patient-reported symptoms into urgency tiers (non_emergency, moderate, urgent). The knowledge base contains 35 paragraphs covering common symptoms, emergency warning signs, condition-specific guidance, first aid, and chronic disease management — giving the RAG system factual grounding when answering triage-related queries.

- 35 paragraphs × ~100–180 words each ≈ 4,500+ words total
- Topics: emergency symptoms, respiratory illness, cardiac events, digestive issues, neurological symptoms, musculoskeletal injury, skin conditions, pediatric concerns, mental health, first aid, medications

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# KNOWLEDGE BASE — 35 medical triage paragraphs
# Each paragraph is a complete, standalone unit covering one topic.
# ─────────────────────────────────────────────────────────────────────────────

PARAGRAPHS = [
    # ── EMERGENCY / CALL 911 ──────────────────────────────────────────────
    """When to Call 911 — Chest Pain and Heart Attack Signs
Chest pain lasting more than a few minutes, or pain that spreads to your arm, shoulder, jaw, or back, may signal a heart attack. Other warning signs include shortness of breath, cold sweats, nausea, and lightheadedness. These symptoms can occur together or separately. Women are more likely than men to experience atypical symptoms such as jaw pain, nausea, or extreme fatigue instead of classic chest pressure. If you or someone nearby experiences these symptoms, call 911 immediately. Do not drive yourself to the hospital. Chew an aspirin (325 mg) if you are not allergic while waiting for emergency services. Time is critical — every minute without treatment increases the risk of permanent heart damage.""",

    """Stroke Recognition — Act FAST
A stroke occurs when blood flow to part of the brain is blocked or a blood vessel bursts. The FAST acronym helps identify stroke symptoms: Face drooping (one side of the face droops or is numb), Arm weakness (one arm is weak or numb, drifts downward when raised), Speech difficulty (slurred or strange speech, inability to speak or understand), Time to call 911. Additional symptoms include sudden severe headache with no known cause, sudden vision problems in one or both eyes, sudden trouble walking or loss of balance, and sudden confusion. Stroke is a medical emergency — treatment with clot-busting medication must happen within 3 to 4.5 hours of symptom onset for the best outcome. Do not wait to see if symptoms improve.""",

    """Severe Allergic Reactions — Anaphylaxis
Anaphylaxis is a life-threatening allergic reaction that can occur within seconds or minutes of exposure to an allergen, such as peanuts, bee stings, or certain medications. Symptoms include throat swelling that can block the airway, difficulty breathing, wheezing, a sudden drop in blood pressure, rapid pulse, dizziness, fainting, skin rash, and nausea. If you have a known severe allergy and carry an epinephrine auto-injector (EpiPen), use it immediately and then call 911. Even if symptoms improve after using epinephrine, emergency evaluation is required because symptoms can return hours later in a biphasic reaction. Never assume the reaction is over after initial improvement.""",

    """Severe Bleeding and Traumatic Injuries
Uncontrolled bleeding from a deep cut, puncture wound, or traumatic injury can become life-threatening within minutes. Apply firm, continuous pressure using a clean cloth or bandage. Do not remove the cloth once applied — add more material on top if soaking through. If the bleeding is on a limb and cannot be controlled, a tourniquet applied 2 to 3 inches above the wound can be lifesaving. Call 911 for injuries that involve large blood vessels, are located near the neck or abdomen, or result from high-force trauma such as car accidents or falls from height. Maintain pressure and keep the person calm and still until emergency services arrive.""",

    """Breathing Emergencies — When Breathing Is Compromised
Sudden severe shortness of breath, gasping, choking, or the inability to speak in full sentences are emergency symptoms requiring an immediate 911 call. Possible causes include severe asthma attack, pulmonary embolism (blood clot in the lungs), pneumothorax (collapsed lung), acute heart failure, or foreign body obstruction. Blue or gray discoloration of the lips or fingernails (cyanosis) indicates dangerously low oxygen levels. For choking, perform the Heimlich maneuver if the person cannot cough or speak. For an unconscious, non-breathing adult, begin CPR and use an automated external defibrillator (AED) if available. These situations require professional intervention — do not delay calling for help.""",

    # ── URGENT CARE — SAME DAY EVALUATION ────────────────────────────────
    """High Fever in Adults — When to Seek Urgent Care
A fever is defined as a body temperature above 100.4°F (38°C). In otherwise healthy adults, fevers up to 103°F are often managed at home with rest, fluids, and over-the-counter antipyretics such as acetaminophen or ibuprofen. However, you should seek urgent medical evaluation if your fever exceeds 103°F, lasts more than three days, or is accompanied by stiff neck, severe headache, confusion, rash, difficulty breathing, or chest or abdominal pain. These symptoms may indicate serious infections such as meningitis, pneumonia, or sepsis. Immunocompromised individuals, the elderly, and those with chronic conditions should seek evaluation at lower fever thresholds than otherwise healthy adults.""",

    """Severe Headache — Distinguishing Migraine from Emergency
Most headaches are caused by tension, dehydration, or migraine and resolve with rest and pain relief. However, some headaches signal dangerous conditions. Seek emergency care immediately for a headache described as the 'worst headache of your life' that comes on suddenly, as it may indicate a subarachnoid hemorrhage. Also seek emergency care for a headache accompanied by fever and stiff neck (meningitis), sudden visual loss, weakness or numbness in the face or limbs, difficulty speaking, or loss of consciousness. A migraine with aura, while painful, is generally not dangerous, but new or unusual headache patterns in someone over 50 warrant prompt evaluation to rule out intracranial pathology.""",

    """Urinary Tract Infections — Symptoms and When to Act
Urinary tract infections (UTIs) are caused by bacteria entering the urinary system and are more common in women than men. Typical symptoms include burning or pain during urination, frequent urge to urinate, cloudy or foul-smelling urine, and pelvic discomfort. A simple lower UTI can often be confirmed through a urine test and treated with a short course of antibiotics. If symptoms progress to include fever, chills, flank pain (side or back pain), nausea, and vomiting, the infection may have spread to the kidneys — a condition called pyelonephritis that requires prompt medical attention and stronger antibiotic therapy. Untreated kidney infections can lead to sepsis, so do not delay evaluation if upper tract symptoms appear.""",

    """Abdominal Pain — Causes and When to Worry
Abdominal pain is one of the most common reasons people visit urgent care or the emergency department. Mild cramping from gas, indigestion, or constipation typically resolves on its own. However, severe pain, pain that wakes you from sleep, pain localized to the lower right abdomen (possible appendicitis), pain accompanied by vomiting blood or blood in the stool, sudden severe pain after eating (possible pancreatitis or gallbladder attack), or abdominal pain combined with fever all warrant prompt evaluation. Appendicitis, in particular, is a surgical emergency — the classic presentation is pain that starts around the navel and migrates to the lower right with nausea and low-grade fever.""",

    """Eye Injuries and Sudden Vision Changes
Any sudden, painless loss of vision in one or both eyes, flashes of light, a sudden shower of floaters, or a curtain-like shadow across your visual field may indicate a retinal detachment or other serious eye emergency requiring immediate evaluation by an ophthalmologist. Chemical splashes to the eye require immediate flushing with water for at least 15 to 20 minutes before seeking care. Blunt trauma to the eye, especially if accompanied by pain, double vision, or bleeding visible inside the eye, also warrants urgent evaluation. Never rub an eye after a chemical or foreign body exposure. Delaying treatment of a retinal detachment by even a few hours can result in permanent vision loss.""",

    # ── MODERATE — SCHEDULE A VISIT OR TELEHEALTH ────────────────────────
    """Respiratory Infections — Cold vs. Flu vs. COVID-19
Upper respiratory infections are among the most frequent reasons patients seek medical care. The common cold is typically caused by rhinovirus and presents with runny nose, sneezing, sore throat, and mild cough. Influenza (flu) usually has a more abrupt onset with high fever, severe body aches, fatigue, and dry cough. COVID-19 can present similarly to the flu but may also cause loss of smell or taste, persistent shortness of breath, and varied systemic symptoms. Most healthy adults recover from mild respiratory infections with rest, fluids, and symptomatic treatment. Antiviral medications such as oseltamivir (Tamiflu) for influenza are most effective when started within 48 hours of symptom onset.""",

    """Ear Pain and Ear Infections
Ear pain (otalgia) is a common complaint in both children and adults. In children, acute otitis media (middle ear infection) is the most frequent cause, often following a cold. Symptoms include ear pain, tugging at the ear, fever, and irritability in young children. In adults, ear pain more commonly results from outer ear infection (swimmer's ear), ear wax buildup, temporomandibular joint (TMJ) dysfunction, or referred pain from dental issues. Viral middle ear infections often resolve without antibiotics; however, bacterial infections confirmed by a healthcare provider typically require antibiotic treatment. If you experience sudden hearing loss, severe dizziness or vertigo, or discharge from the ear, seek prompt medical evaluation.""",

    """Ankle and Knee Sprains — Home Treatment and When to See a Doctor
Ligament sprains are common injuries, especially among active individuals. Mild to moderate sprains can usually be managed at home using the RICE method: Rest (avoid weight bearing), Ice (apply for 20 minutes every 2 hours for the first 48 hours), Compression (use an elastic bandage), and Elevation (keep the limb raised above heart level). Over-the-counter NSAIDs such as ibuprofen help reduce pain and inflammation. Seek evaluation if you heard a 'pop' at the time of injury, cannot bear weight on the joint, notice significant swelling or bruising, or if pain does not improve within 3 to 5 days — these findings may indicate a ligament tear or fracture requiring imaging.""",

    """Back Pain — Common Causes and Red Flag Symptoms
Low back pain affects approximately 80% of adults at some point in their lives. The majority of cases are mechanical in nature, caused by muscle strain, poor posture, or herniated disc, and improve with time, gentle movement, and over-the-counter analgesics. Ice or heat applied to the affected area can provide relief. However, back pain accompanied by any of the following requires prompt medical evaluation: loss of bladder or bowel control (possible cauda equina syndrome, a surgical emergency), pain that radiates down one or both legs with numbness or tingling, fever suggesting infection such as spinal epidural abscess, unexplained weight loss suggesting malignancy, or history of significant trauma such as a fall from height.""",

    """Skin Rashes — Common Types and Warning Signs
Skin rashes have many causes, including allergic reactions, viral infections, bacterial infections, and inflammatory conditions such as eczema or psoriasis. Most rashes are benign and resolve with topical hydrocortisone, antihistamines, or time. However, certain rashes warrant prompt evaluation. A non-blanching petechial or purpuric rash — small red or purple spots that do not fade when pressed — can indicate a serious infection such as meningococcemia and requires emergency care. A rash accompanied by systemic symptoms such as fever, joint pain, or lymph node swelling should be evaluated. Painful blisters in a dermatomal distribution (band-like pattern on one side of the body) typically indicate shingles and benefit from early antiviral therapy.""",

    # ── RESPIRATORY CONDITIONS ────────────────────────────────────────────
    """Asthma Management and Acute Attacks
Asthma is a chronic inflammatory condition of the airways characterized by episodes of wheezing, shortness of breath, chest tightness, and cough, often triggered by allergens, exercise, cold air, or respiratory infections. Patients with asthma should work with their provider to maintain an asthma action plan. During a mild attack, use a short-acting bronchodilator (rescue inhaler, typically albuterol) as prescribed and monitor symptoms. If symptoms do not improve after using the rescue inhaler, if you require the inhaler more than every 4 hours, if you cannot speak in full sentences, or if lips or fingernails turn blue, seek emergency care immediately. Proper use of controller inhalers (inhaled corticosteroids) daily significantly reduces the frequency of acute attacks.""",

    """Pneumonia — Symptoms, Risk Factors, and Treatment
Pneumonia is an infection of the lung tissue caused by bacteria, viruses, or fungi, leading to inflammation and fluid accumulation in the air sacs. Symptoms include fever (often with chills and sweating), productive cough with colored mucus, pleuritic chest pain (sharp pain that worsens with breathing), shortness of breath, fatigue, and sometimes confusion in elderly patients. Bacterial pneumonia is typically treated with antibiotics; viral pneumonia requires supportive care. Risk factors include advanced age, smoking, immunosuppression, and chronic lung disease. Pneumococcal and influenza vaccines reduce pneumonia risk. Patients who are elderly, immunocompromised, or have severe symptoms such as low oxygen saturation may require hospitalization for IV antibiotics and supplemental oxygen.""",

    # ── CARDIAC AND CIRCULATORY ───────────────────────────────────────────
    """Hypertension — Understanding High Blood Pressure
Hypertension (high blood pressure) is defined as consistently elevated blood pressure readings at or above 130/80 mmHg. It is often called the 'silent killer' because most people experience no symptoms until blood pressure reaches dangerously high levels. Chronic uncontrolled hypertension damages blood vessels and organs, increasing the risk of heart attack, stroke, kidney disease, and heart failure. A single elevated reading does not diagnose hypertension; consistent readings over multiple visits are required. Lifestyle modifications including weight loss, dietary sodium reduction (DASH diet), regular aerobic exercise, limiting alcohol, and quitting smoking can significantly lower blood pressure. When lifestyle changes are insufficient, antihypertensive medications are prescribed. A blood pressure above 180/120 mmHg with symptoms such as headache or chest pain is a hypertensive emergency.""",

    """Palpitations — When an Irregular Heartbeat Needs Evaluation
Palpitations are the sensation of an unusually rapid, strong, or irregular heartbeat. They can be felt in the chest, throat, or neck and may be described as fluttering, pounding, or skipping a beat. Most palpitations are benign, caused by caffeine, stress, dehydration, or harmless extra beats called premature ventricular contractions (PVCs). However, palpitations warrant evaluation when accompanied by dizziness, lightheadedness, fainting, shortness of breath, or chest pain, or when they occur in individuals with known heart disease, a history of fainting, or a family history of sudden cardiac death. A 12-lead ECG and Holter monitor can capture the rhythm and guide further management. Atrial fibrillation, detected on ECG, carries a significant stroke risk and requires anticoagulation therapy.""",

    # ── DIGESTIVE AND GASTROINTESTINAL ────────────────────────────────────
    """Nausea, Vomiting, and Diarrhea — Acute Gastroenteritis
Acute gastroenteritis, commonly called the 'stomach flu,' is usually caused by viral pathogens such as norovirus or rotavirus and is highly contagious. Symptoms include sudden onset nausea, vomiting, diarrhea, abdominal cramping, and sometimes low-grade fever. The illness typically resolves within 1 to 3 days. The primary concern is dehydration, especially in young children, the elderly, and immunocompromised individuals. Treatment focuses on oral rehydration with clear fluids, electrolyte solutions, and gradual reintroduction of a bland diet (BRAT: bananas, rice, applesauce, toast). Seek medical evaluation if you cannot keep any fluids down for more than 24 hours, show signs of severe dehydration (extreme thirst, dark urine, dizziness), have bloody stool, or have fever above 102°F.""",

    """Acid Reflux and GERD
Gastroesophageal reflux disease (GERD) occurs when stomach acid frequently flows back into the esophagus, causing irritation. Common symptoms include heartburn — a burning sensation in the chest that often worsens after eating, lying down, or bending — as well as regurgitation of food or sour liquid, difficulty swallowing, and chronic cough or hoarseness. Lifestyle changes that can reduce GERD symptoms include eating smaller meals, avoiding lying down within 2 to 3 hours of eating, elevating the head of the bed, reducing trigger foods such as caffeine, fatty foods, citrus, and spicy foods, losing weight if overweight, and quitting smoking. Over-the-counter antacids, H2 blockers, and proton pump inhibitors (PPIs) are effective treatments. Persistent or worsening symptoms may indicate esophagitis or Barrett's esophagus requiring endoscopy.""",

    # ── NEUROLOGICAL ──────────────────────────────────────────────────────
    """Migraines — Diagnosis and Treatment
Migraine is a neurological condition characterized by recurrent headaches, typically moderate to severe in intensity, often unilateral (one-sided), pulsating in quality, and lasting 4 to 72 hours. They are frequently accompanied by nausea, vomiting, and sensitivity to light and sound. About 25% of migraine sufferers experience an aura — neurological symptoms such as visual disturbances (zigzag lines, blind spots), numbness, or speech difficulty that precede the headache by 20 to 60 minutes. Treatment includes both acute medications such as triptans or NSAIDs and preventive therapy such as beta-blockers, topiramate, or CGRP antagonists for frequent migraines. Identifying and avoiding personal triggers such as stress, certain foods, irregular sleep, or hormonal changes can reduce migraine frequency significantly.""",

    """Dizziness and Vertigo — Causes and Management
Dizziness is a broad term encompassing lightheadedness, presyncope (feeling about to faint), and vertigo (false sensation of spinning). Benign paroxysmal positional vertigo (BPPV) is the most common cause of true vertigo, caused by displaced calcium crystals in the inner ear canals. BPPV typically presents as brief (seconds to minutes) episodes of intense spinning sensation triggered by changes in head position, such as rolling over in bed. The Epley maneuver, a series of specific head movements, effectively repositions the crystals and resolves BPPV in most cases. Other causes of dizziness include orthostatic hypotension (standing up too quickly), Meniere's disease, vestibular neuritis, medication side effects, and central causes such as cerebellar stroke that require emergency evaluation.""",

    # ── MUSCULOSKELETAL ───────────────────────────────────────────────────
    """Fractures — Recognition and First Aid
A fracture is a break in a bone caused by trauma, overuse (stress fracture), or weakening from conditions such as osteoporosis. Common signs of a fracture include pain, swelling, bruising, deformity or abnormal positioning of the limb, tenderness at the fracture site, and inability to bear weight or use the affected area. For a suspected fracture, immobilize the injured area in the position found using a splint or improvised support, apply ice wrapped in cloth to reduce swelling, elevate if possible, and seek medical evaluation for X-ray confirmation. Open fractures (where bone is visible through the skin) and fractures with neurovascular compromise (loss of sensation, absent pulse below injury) are surgical emergencies. Never attempt to straighten a suspected fracture.""",

    """Joint Pain and Arthritis — Osteoarthritis vs. Rheumatoid Arthritis
Arthritis encompasses over 100 different conditions affecting the joints. Osteoarthritis (OA) is the most common form, resulting from wear-and-tear breakdown of cartilage. It typically affects weight-bearing joints such as the knees, hips, and spine, with symptoms of pain and stiffness that worsen with activity and improve with rest. Rheumatoid arthritis (RA) is an autoimmune condition that causes inflammation in the joint lining, leading to swelling, warmth, morning stiffness lasting more than one hour, and potential joint deformity. RA typically affects joints symmetrically and may have systemic features. Management of OA includes exercise, physical therapy, weight management, and analgesics. RA requires disease-modifying antirheumatic drugs (DMARDs) such as methotrexate to slow disease progression.""",

    # ── SKIN CONDITIONS ───────────────────────────────────────────────────
    """Wound Care — Cuts, Lacerations, and Wound Infection
Proper wound care reduces the risk of infection and promotes healing. For minor cuts and lacerations, clean the wound under running water for 5 minutes, apply antibiotic ointment, and cover with a sterile bandage. Change the dressing daily or when wet or dirty. Signs of wound infection include increasing redness, warmth, swelling, or pain beyond the wound edges; purulent discharge; fever; or red streaks extending from the wound suggesting cellulitis or lymphangitis. A wound that is deep, gaping (greater than 0.5 cm), located on the face, or results from an animal or human bite should be evaluated by a healthcare provider for possible suturing, irrigation, or antibiotic treatment. Ensure tetanus vaccination is current for any puncture wound or contaminated laceration.""",

    """Burns — Classification and Treatment
Burns are classified by depth of tissue injury. First-degree burns affect only the outer skin layer (epidermis) and present with redness and pain without blisters — most sunburns are first-degree. Second-degree burns involve the epidermis and dermis, causing blisters, intense pain, and a wet appearance. Third-degree burns destroy all skin layers, appearing white, brown, or black and are paradoxically painless because nerve endings are destroyed. Minor burns (small area, first or superficial second-degree) are treated by cooling with cool running water for 20 minutes, covering with a non-stick sterile dressing, and analgesics. Seek emergency care for any burn involving the face, hands, feet, or genitals; burns larger than 3 inches; deep second or third-degree burns; burns from electrical sources or inhalation; or burns in children or the elderly.""",

    # ── PEDIATRIC CONCERNS ────────────────────────────────────────────────
    """Fever in Children — Guidelines for Parents
Fever is a common symptom in children and usually indicates the immune system is responding to infection. For infants under 3 months, any fever of 100.4°F or higher is a medical emergency requiring immediate evaluation because bacterial infections can progress rapidly in newborns. For children 3 to 6 months, fever above 102°F warrants a call to the pediatrician. Older children with fever can usually be managed at home with age-appropriate dosing of acetaminophen or ibuprofen (not aspirin), encouragement of fluids, and rest. Contact a healthcare provider if the child's fever lasts more than 2 to 3 days, is accompanied by stiff neck, severe headache, persistent vomiting, rash, or extreme irritability, or if the child appears very unwell regardless of the exact temperature reading.""",

    """Common Childhood Illnesses — Croup, RSV, and Strep Throat
Croup is a viral infection of the upper airway causing swelling around the voice box and a distinctive barking cough, often worse at night. Cool night air or a steamy bathroom can temporarily relieve symptoms. Seek emergency care if the child has stridor (high-pitched sound when breathing in at rest) or significant difficulty breathing. Respiratory syncytial virus (RSV) is the most common cause of bronchiolitis in infants under 2 and presents with wheezing and breathing difficulty. Streptococcal pharyngitis (strep throat) presents with sudden sore throat, fever, swollen lymph nodes, and tonsillar exudate without cough. A rapid strep test confirms the diagnosis and antibiotics are required to prevent rheumatic fever. Unlike viral sore throat, strep requires antibiotic treatment.""",

    # ── MENTAL HEALTH ─────────────────────────────────────────────────────
    """Mental Health Crisis — When to Seek Immediate Help
A mental health crisis is any situation in which a person's behavior puts them at risk of harming themselves or others, or when the person is in significant distress and unable to function. Indicators of crisis include expressing suicidal or homicidal thoughts, giving away prized possessions, recent significant loss or trauma, psychotic symptoms such as hearing voices or paranoid beliefs, or severe agitation or aggression. If someone expresses intent to harm themselves or others, call 911 or take them to the nearest emergency department immediately. The 988 Suicide and Crisis Lifeline (call or text 988 in the US) provides immediate confidential support. Do not leave a person in crisis alone, and do not minimize their distress or expressions of suicidal ideation.""",

    """Anxiety and Panic Attacks — Symptoms and Management
Anxiety disorders are among the most common mental health conditions, affecting over 40 million adults in the United States. Generalized anxiety disorder involves persistent, excessive worry about multiple areas of life. Panic disorder involves recurrent unexpected panic attacks — intense surges of fear accompanied by physical symptoms including palpitations, chest pain, shortness of breath, dizziness, tingling sensations, sweating, trembling, and a sense of impending doom. Panic attacks typically peak within 10 minutes and resolve within 30 minutes. Because the physical symptoms can mimic heart attack, patients often seek emergency care. Management includes cognitive-behavioral therapy (CBT), which is highly effective, as well as medications such as SSRIs, SNRIs, or short-term benzodiazepines. Diaphragmatic breathing and grounding techniques can abort or reduce the severity of panic attacks.""",

    # ── FIRST AID AND MEDICATIONS ────────────────────────────────────────
    """Medication Safety — Common Over-the-Counter Drug Interactions
Over-the-counter (OTC) medications are generally safe when used as directed, but certain combinations carry significant risks. Acetaminophen (Tylenol) is found in many cold and flu combination products; accidentally taking multiple products containing acetaminophen can lead to toxic doses causing liver damage. NSAIDs such as ibuprofen and naproxen should be used cautiously by those with kidney disease, heart disease, or peptic ulcer history. Aspirin should not be given to children under 18 due to the risk of Reye's syndrome. Antihistamines such as diphenhydramine (Benadryl) cause sedation and should not be combined with alcohol or other sedating medications. Always read labels, check active ingredients in combination products, and consult a pharmacist if you take prescription medications to avoid drug interactions.""",

    """Diabetes Management and Hypoglycemia
Diabetes is a chronic metabolic condition characterized by elevated blood glucose levels due to insulin deficiency or resistance. Type 1 diabetes requires insulin therapy; Type 2 diabetes is initially managed with lifestyle changes and oral medications. Monitoring blood glucose levels is central to diabetes management. Hypoglycemia (low blood sugar, below 70 mg/dL) can occur in diabetic patients taking insulin or certain oral medications and presents with shakiness, sweating, confusion, rapid heartbeat, and hunger. Mild to moderate hypoglycemia is treated with the '15-15 rule': consume 15 grams of fast-acting carbohydrates (e.g., 4 glucose tablets, half a cup of juice), wait 15 minutes, and recheck blood glucose. Severe hypoglycemia with loss of consciousness requires glucagon injection or emergency services.""",

    """Dehydration — Recognizing and Treating Fluid Deficit
Dehydration occurs when fluid loss exceeds fluid intake, disrupting normal body functions. It can result from vomiting, diarrhea, excessive sweating, fever, inadequate fluid intake, or conditions such as diabetes. Mild dehydration symptoms include thirst, dry mouth, decreased urine output, and dark-colored urine. Moderate to severe dehydration presents with dizziness, lightheadedness, rapid heartbeat, headache, confusion, and in severe cases, fainting or shock. Treatment for mild to moderate dehydration is oral rehydration with water and electrolyte solutions. Sports drinks may be appropriate for exercise-induced dehydration, but sugary beverages worsen diarrhea-associated dehydration. Infants, the elderly, and individuals with severe vomiting who cannot tolerate oral fluids may require intravenous fluid replacement in a medical setting.""",

    """Heat-Related Illnesses — Heat Exhaustion and Heat Stroke
Heat-related illnesses occur when the body cannot cool itself adequately, most commonly in hot, humid environments or during strenuous exercise. Heat exhaustion presents with heavy sweating, cold and clammy skin, fast and weak pulse, nausea, muscle cramps, tiredness, weakness, and headache. Move the person to a cool environment, loosen clothing, apply cool wet cloths, and have them sip cool water. Heat stroke is a life-threatening emergency where the core body temperature rises above 104°F and the sweating mechanism fails. Symptoms include hot, dry, red skin, rapid strong pulse, confusion, and possible unconsciousness. Call 911 immediately and begin aggressive cooling with ice packs to the neck, armpits, and groin while waiting for emergency services. Heat stroke can cause organ failure and death without immediate treatment.""",

    """Preventive Health — Vaccinations, Screenings, and Annual Exams
Preventive healthcare is the cornerstone of reducing chronic disease burden and detecting conditions at earlier, more treatable stages. Adults should maintain immunization schedules including annual influenza vaccine, Tdap booster every 10 years, COVID-19 vaccines as recommended, pneumococcal vaccine for those 65 and older, shingles vaccine for adults 50 and older, and HPV vaccine for eligible individuals. Cancer screenings include colorectal cancer screening starting at age 45, mammography for women starting at age 40 to 50 depending on guidelines followed, cervical cancer screening with Pap smears, and lung cancer screening with low-dose CT for high-risk current and former smokers. Annual wellness visits include blood pressure, cholesterol, and glucose screening, allowing for early intervention before conditions become serious.""",

    """When to Use the Emergency Room vs. Urgent Care vs. Primary Care
Choosing the appropriate level of care saves time, money, and healthcare system resources. The emergency room should be used for life-threatening or severe conditions: chest pain, stroke symptoms, severe breathing difficulty, major trauma, uncontrolled bleeding, severe allergic reactions, and loss of consciousness. Urgent care centers are appropriate for conditions requiring same-day evaluation but that are not immediately life-threatening: moderate fever, minor fractures, sprains, lacerations requiring sutures, urinary tract infections, and ear infections when your primary care provider is unavailable. Primary care visits are appropriate for ongoing management of chronic conditions, annual physicals, routine prescription refills, non-urgent new symptoms, mental health follow-up, and preventive care. Telehealth is increasingly available for many urgent and primary care concerns, offering convenient same-day access.""",
]

print(f"Knowledge base loaded: {len(PARAGRAPHS)} paragraphs")
total_words = sum(len(p.split()) for p in PARAGRAPHS)
avg_words = total_words / len(PARAGRAPHS)
print(f"Total words: {total_words}")
print(f"Average words per paragraph: {avg_words:.1f}")

Knowledge base loaded: 37 paragraphs
Total words: 4506
Average words per paragraph: 121.8


---
### Step 1.2 — Text Representation & Chunking

We implement **three** chunking strategies. Sentences are never split across chunks. All chunking operates on the paragraph list above.

| Strategy | Description |
|---|---|
| Fixed-Length | Group consecutive paragraphs until a word-count target is reached |
| Overlapping Paragraph | Slide a 2-paragraph window with 1-paragraph stride |
| Hybrid/Strategic | Topic-aware: group semantically related paragraphs by medical category |

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Strategy 1 — Fixed-Length Chunking
#
# Definition: Group consecutive paragraphs into chunks until a cumulative
# word count target (~200 words) is reached. A paragraph is never split
# mid-sentence — it is always included whole in whichever chunk it starts in.
#
# Implementation detail: we accumulate whole paragraphs until adding the next
# would exceed FIXED_WORD_LIMIT, at which point the current buffer is sealed
# as a chunk and a new buffer begins.
# ─────────────────────────────────────────────────────────────────────────────

FIXED_WORD_LIMIT = 200  # target words per chunk

def fixed_length_chunks(paragraphs, word_limit=FIXED_WORD_LIMIT):
    """Group consecutive whole paragraphs up to word_limit words per chunk."""
    chunks = []
    buffer = []
    buffer_words = 0

    for para in paragraphs:
        para_words = len(para.split())
        # If adding this paragraph would exceed the limit and buffer is non-empty, flush
        if buffer and buffer_words + para_words > word_limit:
            chunks.append("\n\n".join(buffer))
            buffer = []
            buffer_words = 0
        buffer.append(para)
        buffer_words += para_words

    if buffer:  # flush remaining
        chunks.append("\n\n".join(buffer))

    return chunks


fixed_chunks = fixed_length_chunks(PARAGRAPHS)
print(f"Strategy 1 — Fixed-Length Chunking")
print(f"  Chunks produced: {len(fixed_chunks)}")
print(f"  Avg words/chunk: {sum(len(c.split()) for c in fixed_chunks)/len(fixed_chunks):.1f}")
print()
for i, chunk in enumerate(fixed_chunks):
    print(f"  Chunk {i+1} ({len(chunk.split())} words): {chunk[:80].strip()!r}...")

Strategy 1 — Fixed-Length Chunking
  Chunks produced: 37
  Avg words/chunk: 121.8

  Chunk 1 (125 words): 'When to Call 911 — Chest Pain and Heart Attack Signs\nChest pain lasting more tha'...
  Chunk 2 (128 words): 'Stroke Recognition — Act FAST\nA stroke occurs when blood flow to part of the bra'...
  Chunk 3 (107 words): 'Severe Allergic Reactions — Anaphylaxis\nAnaphylaxis is a life-threatening allerg'...
  Chunk 4 (113 words): 'Severe Bleeding and Traumatic Injuries\nUncontrolled bleeding from a deep cut, pu'...
  Chunk 5 (109 words): 'Breathing Emergencies — When Breathing Is Compromised\nSudden severe shortness of'...
  Chunk 6 (111 words): 'High Fever in Adults — When to Seek Urgent Care\nA fever is defined as a body tem'...
  Chunk 7 (114 words): 'Severe Headache — Distinguishing Migraine from Emergency\nMost headaches are caus'...
  Chunk 8 (123 words): 'Urinary Tract Infections — Symptoms and When to Act\nUrinary tract infections (UT'...
  Chunk 9 (114 words): 'Abdominal Pai

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Strategy 2 — Overlapping Paragraph Chunking
#
# Definition: A sliding window of 2 consecutive paragraphs advances by 1
# paragraph at a time, so adjacent chunks share exactly 1 paragraph of overlap.
#
# Example with 4 paragraphs [P1, P2, P3, P4]:
#   Chunk 1 → P1 + P2
#   Chunk 2 → P2 + P3
#   Chunk 3 → P3 + P4
#
# Overlap purpose: ensures that information near a paragraph boundary is
# represented in two chunks, improving retrieval recall for queries that
# straddle topic transitions.
#
# No sentence is ever split — paragraphs are always included whole.
# ─────────────────────────────────────────────────────────────────────────────

WINDOW_SIZE = 2   # number of paragraphs per chunk
STRIDE = 1        # paragraphs advanced per step (overlap = WINDOW_SIZE - STRIDE)

def overlapping_chunks(paragraphs, window=WINDOW_SIZE, stride=STRIDE):
    """Sliding window over paragraphs with (window - stride) paragraphs of overlap."""
    chunks = []
    i = 0
    while i < len(paragraphs):
        window_paras = paragraphs[i : i + window]
        chunks.append("\n\n".join(window_paras))
        if i + window >= len(paragraphs):  # last window reached
            break
        i += stride
    return chunks


overlap_chunks = overlapping_chunks(PARAGRAPHS)
print(f"Strategy 2 — Overlapping Paragraph Chunking (window={WINDOW_SIZE}, stride={STRIDE})")
print(f"  Chunks produced: {len(overlap_chunks)}")
print(f"  Avg words/chunk: {sum(len(c.split()) for c in overlap_chunks)/len(overlap_chunks):.1f}")
print(f"  Overlap: {WINDOW_SIZE - STRIDE} paragraph(s) between consecutive chunks")
print()
for i, chunk in enumerate(overlap_chunks[:5]):
    print(f"  Chunk {i+1} ({len(chunk.split())} words): {chunk[:80].strip()!r}...")
print(f"  ... (showing first 5 of {len(overlap_chunks)})")

Strategy 2 — Overlapping Paragraph Chunking (window=2, stride=1)
  Chunks produced: 36
  Avg words/chunk: 243.2
  Overlap: 1 paragraph(s) between consecutive chunks

  Chunk 1 (253 words): 'When to Call 911 — Chest Pain and Heart Attack Signs\nChest pain lasting more tha'...
  Chunk 2 (235 words): 'Stroke Recognition — Act FAST\nA stroke occurs when blood flow to part of the bra'...
  Chunk 3 (220 words): 'Severe Allergic Reactions — Anaphylaxis\nAnaphylaxis is a life-threatening allerg'...
  Chunk 4 (222 words): 'Severe Bleeding and Traumatic Injuries\nUncontrolled bleeding from a deep cut, pu'...
  Chunk 5 (220 words): 'Breathing Emergencies — When Breathing Is Compromised\nSudden severe shortness of'...
  ... (showing first 5 of 36)


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Strategy 3 — Hybrid / Strategic Chunking
#
# Definition: Paragraphs are manually grouped by medical topic category.
# All paragraphs within the same category are combined into one chunk,
# ensuring semantic coherence within each chunk.
#
# Categories:
#   - Emergency / Call 911 (paragraphs 0–4)
#   - Urgent Care — Same-Day Evaluation (paragraphs 5–9)
#   - Moderate — Schedule a Visit (paragraphs 10–14)
#   - Respiratory Conditions (paragraphs 15–16)
#   - Cardiac and Circulatory (paragraphs 17–18)
#   - Digestive / GI (paragraphs 19–20)
#   - Neurological (paragraphs 21–22)
#   - Musculoskeletal (paragraphs 23–24)
#   - Skin Conditions (paragraphs 25–26)
#   - Pediatric Concerns (paragraphs 27–28)
#   - Mental Health (paragraphs 29–30)
#   - First Aid and Medications (paragraphs 31–34)
#
# Rationale: A user query about "chest pain" or "heart attack" will retrieve
# a chunk covering all emergency cardiac information rather than only a single
# isolated paragraph, providing richer context to the LLM.
# ─────────────────────────────────────────────────────────────────────────────

TOPIC_GROUPS = [
    ("Emergency / Call 911",               list(range(0, 5))),
    ("Urgent Care — Same-Day Evaluation",  list(range(5, 10))),
    ("Moderate — Schedule a Visit",        list(range(10, 15))),
    ("Respiratory Conditions",             list(range(15, 17))),
    ("Cardiac and Circulatory",            list(range(17, 19))),
    ("Digestive / Gastrointestinal",       list(range(19, 21))),
    ("Neurological",                       list(range(21, 23))),
    ("Musculoskeletal",                    list(range(23, 25))),
    ("Skin Conditions",                    list(range(25, 27))),
    ("Pediatric Concerns",                 list(range(27, 29))),
    ("Mental Health",                      list(range(29, 31))),
    ("First Aid and Medications",          list(range(31, 35))),
]

def strategic_chunks(paragraphs, topic_groups):
    """Group paragraphs by topic category into semantically coherent chunks."""
    chunks = []
    chunk_labels = []
    for label, indices in topic_groups:
        group_paras = [paragraphs[i] for i in indices if i < len(paragraphs)]
        chunks.append("\n\n".join(group_paras))
        chunk_labels.append(label)
    return chunks, chunk_labels


hybrid_chunks, hybrid_labels = strategic_chunks(PARAGRAPHS, TOPIC_GROUPS)
print(f"Strategy 3 — Hybrid / Strategic (Topic-Aware) Chunking")
print(f"  Chunks produced: {len(hybrid_chunks)}")
print(f"  Avg words/chunk: {sum(len(c.split()) for c in hybrid_chunks)/len(hybrid_chunks):.1f}")
print()
for label, chunk in zip(hybrid_labels, hybrid_chunks):
    print(f"  [{label}]  {len(chunk.split())} words")

Strategy 3 — Hybrid / Strategic (Topic-Aware) Chunking
  Chunks produced: 12
  Avg words/chunk: 354.2

  [Emergency / Call 911]  582 words
  [Urgent Care — Same-Day Evaluation]  581 words
  [Moderate — Schedule a Visit]  591 words
  [Respiratory Conditions]  236 words
  [Cardiac and Circulatory]  253 words
  [Digestive / Gastrointestinal]  247 words
  [Neurological]  241 words
  [Musculoskeletal]  245 words
  [Skin Conditions]  261 words
  [Pediatric Concerns]  256 words
  [Mental Health]  262 words
  [First Aid and Medications]  496 words


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Chunking Summary
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

summary_data = {
    "Strategy": ["Fixed-Length", "Overlapping Paragraph", "Hybrid/Strategic"],
    "# Chunks": [len(fixed_chunks), len(overlap_chunks), len(hybrid_chunks)],
    "Avg Words/Chunk": [
        round(sum(len(c.split()) for c in fixed_chunks) / len(fixed_chunks), 1),
        round(sum(len(c.split()) for c in overlap_chunks) / len(overlap_chunks), 1),
        round(sum(len(c.split()) for c in hybrid_chunks) / len(hybrid_chunks), 1),
    ],
    "Min Words": [
        min(len(c.split()) for c in fixed_chunks),
        min(len(c.split()) for c in overlap_chunks),
        min(len(c.split()) for c in hybrid_chunks),
    ],
    "Max Words": [
        max(len(c.split()) for c in fixed_chunks),
        max(len(c.split()) for c in overlap_chunks),
        max(len(c.split()) for c in hybrid_chunks),
    ],
}

df_chunks = pd.DataFrame(summary_data)
print("Chunking Strategy Summary:")
print(df_chunks.to_string(index=False))

# Bundle all strategies into a dict for downstream use
CHUNKING_STRATEGIES = {
    "fixed": fixed_chunks,
    "overlapping": overlap_chunks,
    "hybrid": hybrid_chunks,
}

Chunking Strategy Summary:
             Strategy  # Chunks  Avg Words/Chunk  Min Words  Max Words
         Fixed-Length        37            121.8        107        133
Overlapping Paragraph        36            243.2        220        263
     Hybrid/Strategic        12            354.2        236        591


---
### Step 1.3 — Embedding Pipeline

We use three `sentence-transformers` models of different sizes:

| Size | Model | Embedding Dimensions |
|---|---|---|
| Small | `all-MiniLM-L6-v2` | 384 |
| Medium | `all-mpnet-base-v2` | 768 |
| Large | `all-roberta-large-v1` | 1024 |

Each chunk is embedded and stored in a dictionary keyed by `(embedding_model, chunking_strategy)`.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Embedding Model Definitions
# ─────────────────────────────────────────────────────────────────────────────

EMBEDDING_MODELS = {
    "small":  "all-MiniLM-L6-v2",    # 384-dim
    "medium": "all-mpnet-base-v2",    # 768-dim
    "large":  "all-roberta-large-v1", # 1024-dim
}

print("Embedding models to load:")
for size, model_id in EMBEDDING_MODELS.items():
    print(f"  {size:8s}: {model_id}")

Embedding models to load:
  small   : all-MiniLM-L6-v2
  medium  : all-mpnet-base-v2
  large   : all-roberta-large-v1


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Build Vector Stores
#
# For each combination of (embedding_model × chunking_strategy), we:
#   1. Load the sentence-transformer model
#   2. Encode all chunks into dense embedding vectors
#   3. Store the (chunks, embeddings) pair in VECTOR_STORES dict
#
# Storage format:
#   VECTOR_STORES[(embedding_size, chunking_strategy)] = {
#       "chunks":     list[str],
#       "embeddings": np.ndarray of shape (n_chunks, embedding_dim),
#       "model_id":   str,
#   }
# ─────────────────────────────────────────────────────────────────────────────

VECTOR_STORES = {}

for size, model_id in EMBEDDING_MODELS.items():
    print(f"\nLoading embedding model: {model_id} ({size})")
    t0 = time.time()
    embedder = SentenceTransformer(model_id)
    load_time = time.time() - t0
    
    # Report actual embedding dimension
    sample_emb = embedder.encode(["test"])
    dim = sample_emb.shape[1]
    print(f"  Embedding dimension: {dim}  |  Load time: {load_time:.1f}s")

    for strategy_name, chunks in CHUNKING_STRATEGIES.items():
        t1 = time.time()
        embeddings = embedder.encode(chunks, show_progress_bar=False)
        encode_time = time.time() - t1

        key = (size, strategy_name)
        VECTOR_STORES[key] = {
            "chunks":     chunks,
            "embeddings": embeddings,
            "model_id":   model_id,
            "dim":        dim,
        }
        print(f"  [{strategy_name:12s}] {len(chunks):3d} chunks encoded in {encode_time:.2f}s")

print(f"\nVector stores built: {len(VECTOR_STORES)} configurations")


Loading embedding model: all-MiniLM-L6-v2 (small)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10163.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dimension: 384  |  Load time: 5.7s
  [fixed       ]  37 chunks encoded in 1.74s
  [overlapping ]  36 chunks encoded in 0.54s
  [hybrid      ]  12 chunks encoded in 0.15s

Loading embedding model: all-mpnet-base-v2 (medium)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2173.17it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dimension: 768  |  Load time: 79.9s
  [fixed       ]  37 chunks encoded in 2.58s
  [overlapping ]  36 chunks encoded in 3.57s
  [hybrid      ]  12 chunks encoded in 0.62s

Loading embedding model: all-roberta-large-v1 (large)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4202.77it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-roberta-large-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dimension: 1024  |  Load time: 279.9s
  [fixed       ]  37 chunks encoded in 3.13s
  [overlapping ]  36 chunks encoded in 12.31s
  [hybrid      ]  12 chunks encoded in 1.01s

Vector stores built: 9 configurations


---
### Step 1.4 — Retrieval System

Retrieval uses **cosine similarity** between the query embedding and all chunk embeddings. The top-*k* most similar chunks are returned as context.

Pipeline: `Query string → embed → cosine similarity against all chunks → return top-k chunks`

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrieval Utilities
# ─────────────────────────────────────────────────────────────────────────────

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between vector a and matrix b (n_chunks × dim)."""
    a_norm = a / (np.linalg.norm(a) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
    return b_norm @ a_norm


def retrieve(
    query: str,
    embedding_size: str,
    chunking_strategy: str,
    top_k: int = 3,
    embedder_cache: dict = {},
) -> list[dict]:
    """
    Retrieve the top-k most relevant chunks for a query.

    Returns a list of dicts:
      {
        "rank":  int,
        "score": float,      # cosine similarity
        "chunk": str,        # text of the retrieved chunk
      }
    """
    key = (embedding_size, chunking_strategy)
    store = VECTOR_STORES[key]

    # Cache embedders so we don't reload them for every query
    if embedding_size not in embedder_cache:
        embedder_cache[embedding_size] = SentenceTransformer(store["model_id"])
    embedder = embedder_cache[embedding_size]

    query_emb = embedder.encode([query])[0]
    similarities = cosine_similarity(query_emb, store["embeddings"])

    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = [
        {
            "rank":  rank + 1,
            "score": float(similarities[idx]),
            "chunk": store["chunks"][idx],
        }
        for rank, idx in enumerate(top_indices)
    ]
    return results


# Pre-warm embedder cache so retrieval is faster during experiments
_EMBEDDER_CACHE = {}
for size, model_id in EMBEDDING_MODELS.items():
    _EMBEDDER_CACHE[size] = SentenceTransformer(model_id)
    print(f"Embedder cached: {size} ({model_id})")


def retrieve_cached(query, embedding_size, chunking_strategy, top_k=3):
    """Retrieve using pre-warmed embedder cache."""
    return retrieve(query, embedding_size, chunking_strategy, top_k, _EMBEDDER_CACHE)


print("\nRetrieval system ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7360.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder cached: small (all-MiniLM-L6-v2)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3855.40it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder cached: medium (all-mpnet-base-v2)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4352.62it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-roberta-large-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder cached: large (all-roberta-large-v1)

Retrieval system ready.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Smoke-test retrieval with a sample query
# ─────────────────────────────────────────────────────────────────────────────

TEST_QUERY = "I have chest pain and my left arm feels numb. What should I do?"

print(f"Query: {TEST_QUERY}\n")
print("=" * 70)

for emb_size in ["small", "medium", "large"]:
    for strategy in ["fixed", "overlapping", "hybrid"]:
        t0 = time.time()
        results = retrieve_cached(TEST_QUERY, emb_size, strategy, top_k=2)
        elapsed = time.time() - t0
        print(f"\n[{emb_size:6s} | {strategy:12s}]  retrieval time: {elapsed:.3f}s")
        for r in results:
            preview = r['chunk'][:120].replace('\n', ' ')
            print(f"  Rank {r['rank']}  score={r['score']:.4f}  {preview!r}...")

Query: I have chest pain and my left arm feels numb. What should I do?


[small  | fixed       ]  retrieval time: 5.108s
  Rank 1  score=0.5807  'When to Call 911 — Chest Pain and Heart Attack Signs Chest pain lasting more than a few minutes, or pain that spreads to'...
  Rank 2  score=0.4243  'Palpitations — When an Irregular Heartbeat Needs Evaluation Palpitations are the sensation of an unusually rapid, strong'...

[small  | overlapping ]  retrieval time: 0.019s
  Rank 1  score=0.5434  'When to Call 911 — Chest Pain and Heart Attack Signs Chest pain lasting more than a few minutes, or pain that spreads to'...
  Rank 2  score=0.3727  'Palpitations — When an Irregular Heartbeat Needs Evaluation Palpitations are the sensation of an unusually rapid, strong'...

[small  | hybrid      ]  retrieval time: 0.010s
  Rank 1  score=0.5434  'When to Call 911 — Chest Pain and Heart Attack Signs Chest pain lasting more than a few minutes, or pain that spreads to'...
  Rank 2  score=0.3406  'Hypert

---
### Step 1.5 — Generation Pipeline

We use `llama-3.3-70b-versatile` via Groq — the best-performing model from A7 (95% accuracy on triage classification). The prompt is structured as:

1. **System instruction** — role and behavior guidelines
2. **Retrieved context** — top-k chunks from the vector store
3. **User query** — the patient's question

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Prompt Construction
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a knowledgeable medical triage assistant for the UIUC-Med app.
Your role is to help users understand the urgency of their symptoms and guide them
to appropriate care. You have access to verified medical knowledge provided in the
context below.

Guidelines:
- Base your answer ONLY on the provided context.
- Be clear and concise. Use plain language.
- Always indicate the urgency level: non_emergency, moderate, or urgent.
- If symptoms may be life-threatening, explicitly advise calling 911.
- Never diagnose; recommend appropriate level of care."""


def build_rag_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    """Construct the user message combining retrieved context and query."""
    context_sections = []
    for r in retrieved_chunks:
        context_sections.append(
            f"[Context {r['rank']} | Similarity: {r['score']:.3f}]\n{r['chunk']}"
        )
    context_block = "\n\n---\n\n".join(context_sections)

    return (
        f"MEDICAL KNOWLEDGE CONTEXT:\n\n"
        f"{context_block}\n\n"
        f"---\n\n"
        f"PATIENT QUESTION:\n{query}"
    )


print("Prompt builder ready.")
print(f"Generation model: {GENERATION_MODEL}")

Prompt builder ready.
Generation model: llama-3.3-70b-versatile


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Generation Function
# ─────────────────────────────────────────────────────────────────────────────

def generate_answer(
    query: str,
    embedding_size: str,
    chunking_strategy: str,
    top_k: int = 3,
    max_tokens: int = 400,
    temperature: float = 0.0,
) -> dict:
    """
    Full RAG pipeline: retrieve → prompt → generate.

    Returns:
      {
        "query":            str,
        "embedding_size":   str,
        "chunking_strategy":str,
        "retrieved_chunks": list[dict],
        "prompt":           str,
        "answer":           str,
        "retrieval_time_s": float,
        "generation_time_s":float,
        "total_time_s":     float,
      }
    """
    # Step 1: Retrieve relevant chunks
    t_ret = time.time()
    retrieved = retrieve_cached(query, embedding_size, chunking_strategy, top_k)
    retrieval_time = time.time() - t_ret

    # Step 2: Build prompt
    user_message = build_rag_prompt(query, retrieved)

    # Step 3: Generate answer
    t_gen = time.time()
    response = groq_client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    generation_time = time.time() - t_gen
    answer = response.choices[0].message.content.strip()

    return {
        "query":             query,
        "embedding_size":    embedding_size,
        "chunking_strategy": chunking_strategy,
        "retrieved_chunks":  retrieved,
        "prompt":            user_message,
        "answer":            answer,
        "retrieval_time_s":  round(retrieval_time, 3),
        "generation_time_s": round(generation_time, 3),
        "total_time_s":      round(retrieval_time + generation_time, 3),
    }


print("Generation pipeline ready.")

Generation pipeline ready.


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# End-to-End Demo — Part 1 Smoke Test
# ─────────────────────────────────────────────────────────────────────────────

demo_query = "I have chest pain and my left arm feels numb. What should I do?"
demo_result = generate_answer(
    query=demo_query,
    embedding_size="medium",
    chunking_strategy="hybrid",
    top_k=3,
)

print("=" * 70)
print("QUERY:", demo_result["query"])
print("=" * 70)

print("\n--- RETRIEVED CHUNKS ---")
for r in demo_result["retrieved_chunks"]:
    print(f"\nRank {r['rank']}  |  Cosine Similarity: {r['score']:.4f}")
    print(r["chunk"][:300] + "...")

print("\n" + "=" * 70)
print("GENERATED ANSWER:")
print("=" * 70)
print(demo_result["answer"])

print(f"\nLatency — Retrieval: {demo_result['retrieval_time_s']}s  |  "
      f"Generation: {demo_result['generation_time_s']}s  |  "
      f"Total: {demo_result['total_time_s']}s")

QUERY: I have chest pain and my left arm feels numb. What should I do?

--- RETRIEVED CHUNKS ---

Rank 1  |  Cosine Similarity: 0.5511
When to Call 911 — Chest Pain and Heart Attack Signs
Chest pain lasting more than a few minutes, or pain that spreads to your arm, shoulder, jaw, or back, may signal a heart attack. Other warning signs include shortness of breath, cold sweats, nausea, and lightheadedness. These symptoms can occur to...

Rank 2  |  Cosine Similarity: 0.3727
Hypertension — Understanding High Blood Pressure
Hypertension (high blood pressure) is defined as consistently elevated blood pressure readings at or above 130/80 mmHg. It is often called the 'silent killer' because most people experience no symptoms until blood pressure reaches dangerously high lev...

Rank 3  |  Cosine Similarity: 0.3596
Mental Health Crisis — When to Seek Immediate Help
A mental health crisis is any situation in which a person's behavior puts them at risk of harming themselves or others, or when th

---
## Part 1 Complete

The full RAG pipeline is operational:

| Component | Implementation |
|---|---|
| Knowledge Base | 35 medical triage paragraphs, ~4,500+ words |
| Chunking | Fixed-length, Overlapping paragraph, Hybrid/Strategic |
| Embedding — Small | `all-MiniLM-L6-v2` (384-dim) |
| Embedding — Medium | `all-mpnet-base-v2` (768-dim) |
| Embedding — Large | `all-roberta-large-v1` (1024-dim) |
| Retrieval | Cosine similarity, configurable top-k |
| Generation | `llama-3.3-70b-versatile` via Groq |

Part 2 (experiments across 9 configurations) and Parts 3–4 follow in `rag_analysis.md`.